# NICTO v5 — 7-Brain MoE+MLA Training (Kaggle)

Self-contained notebook: installs dependencies, loads 7-brain architecture,
downloads datasets, and runs training on Kaggle GPU.

In [ ]:
# Install dependencies
!pip install -q torch numpy datasets psutil
import os, sys, json, math, time, random
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# === MultiHeadLatentAttention ===
class MultiHeadLatentAttention(nn.Module):
    def __init__(self, d_model, n_heads, latent_ratio=0.5):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.latent_dim = max(1, int(d_model * latent_ratio))
        self.q_proj = nn.Linear(d_model, self.latent_dim, bias=False)
        self.k_proj = nn.Linear(d_model, self.latent_dim, bias=False)
        self.v_proj = nn.Linear(d_model, self.latent_dim, bias=False)
        self.q_expand = nn.Linear(self.latent_dim, d_model, bias=False)
        self.k_expand = nn.Linear(self.latent_dim, d_model, bias=False)
        self.v_expand = nn.Linear(self.latent_dim, d_model, bias=False)
        self.out = nn.Linear(d_model, d_model, bias=False)
        self.scale = self.head_dim ** -0.5

    def forward(self, x, mask=None):
        B, T, D = x.shape
        q = self.q_expand(self.q_proj(x)).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_expand(self.k_proj(x)).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_expand(self.v_proj(x)).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        attn = (q @ k.transpose(-2, -1)) * self.scale
        if mask is not None:
            attn = attn.masked_fill(mask == 0, float('-inf'))
        attn = F.softmax(attn, dim=-1)
        out = (attn @ v).transpose(1, 2).contiguous().view(B, T, D)
        return self.out(out)

# === EnhancedMoE ===
class EnhancedMoE(nn.Module):
    def __init__(self, d_model, d_ff, n_experts=8, top_k=2):
        super().__init__()
        self.n_experts = n_experts
        self.top_k = top_k
        self.router = nn.Linear(d_model, n_experts, bias=False)
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d_model, d_ff, bias=False),
                nn.GELU(),
                nn.Linear(d_ff, d_model, bias=False),
            ) for _ in range(n_experts)
        ])

    def forward(self, x):
        B, T, D = x.shape
        weights = F.softmax(self.router(x), dim=-1)
        top_w, top_idx = torch.topk(weights, self.top_k, dim=-1)
        top_w = F.softmax(top_w, dim=-1)
        out = torch.zeros_like(x)
        for i in range(self.top_k):
            idx = top_idx[:, :, i]
            w = top_w[:, :, i:i+1]
            for e_idx in range(self.n_experts):
                mask = (idx == e_idx).float().unsqueeze(-1)
                if mask.sum() > 0:
                    expert_out = self.experts[e_idx](x)
                    out += expert_out * w * mask
        return out

# === Brain SubNetwork ===
class SubNetwork(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_model * 2),
            nn.GELU(),
            nn.Linear(d_model * 2, d_model),
        )
        self.confidence = nn.Linear(d_model, 1)

    def forward(self, x):
        return self.net(x), torch.sigmoid(self.confidence(x))

In [ ]:
# === 7 Brains: 10 SubNetworks each ===
class Brain(nn.Module):
    def __init__(self, d_model, n_subnets=10):
        super().__init__()
        self.mla = MultiHeadLatentAttention(d_model, n_heads=4)
        self.moe = EnhancedMoE(d_model, d_model * 4, n_experts=8, top_k=2)
        self.subnets = nn.ModuleList([SubNetwork(d_model) for _ in range(n_subnets)])
        self.router = nn.Linear(d_model, n_subnets)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        x = self.norm(x + self.mla(x))
        x = self.norm(x + self.moe(x))
        route_w = F.softmax(self.router(x.mean(dim=1, keepdim=True)), dim=-1)
        outputs, confidences = [], []
        for i, net in enumerate(self.subnets):
            o, c = net(x)
            rw = route_w[:, :, i:i+1]
            outputs.append(o * rw)
            confidences.append(c * rw)
        return sum(outputs), sum(confidences)

# Build all 7 brains
D_MODEL = 128
brains = nn.ModuleDict({
    name: Brain(D_MODEL, 10)
    for name in ['math','primary','analytical','creative','strategic','knowledge','intuitive']
})

In [ ]:
# === Test forward pass ===
x = torch.randn(2, 16, D_MODEL)
for name, brain in brains.items():
    out, conf = brain(x)
    print(f'{name:12s} | out: {list(out.shape)} | conf: {conf.shape}')
print('All 7 brains forward OK')

In [ ]:
# === Dataset ===
class TextDataset(Dataset):
    def __init__(self, texts, vocab_size=100, max_len=64):
        self.data = []
        for t in texts:
            ids = [hash(c) % vocab_size for c in str(t)]
            for i in range(0, len(ids)-max_len, max_len//2):
                chunk = ids[i:i+max_len]
                if len(chunk) < max_len:
                    chunk = chunk + [0]*(max_len-len(chunk))
                self.data.append(torch.tensor(chunk, dtype=torch.long))
    def __len__(self): return len(self.data)
    def __getitem__(self, i):
        x = self.data[i]
        return x[:-1], x[1:]

# Load training texts from datasets library
try:
    from datasets import load_dataset
    ds = load_dataset('tatsu-lab/alpaca', split='train', streaming=True)
    texts = [ex['instruction'] + ' ' + ex['output'] for ex in ds]
    dataset = TextDataset(texts[:500])
    print(f'Loaded {len(dataset)} training samples')
except Exception as e:
    print(f'Dataset load: {e}')
    texts = ['NICTO AI training example ' + str(i) for i in range(500)]
    dataset = TextDataset(texts)

In [ ]:
# === Training ===
from torch.utils.data import DataLoader
loader = DataLoader(dataset, batch_size=8, shuffle=True)
optimizer = torch.optim.AdamW(brains.parameters(), lr=3e-4)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
brains.to(device)
print(f'Training on {device}')

projection = nn.Linear(D_MODEL, 100).to(device)
criterion = nn.CrossEntropyLoss()

for epoch in range(3):
    total_loss = 0.0
    for inputs, targets in loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        fused_out = torch.zeros(inputs.size(0), inputs.size(1), D_MODEL, device=device)
        for name, brain in brains.items():
            out, _ = brain(fused_out)
            fused_out = fused_out + out
        logits = projection(fused_out)
        loss = criterion(logits.view(-1, 100), targets.view(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(loader)
    print(f'Epoch {epoch+1}/3 | Loss: {avg_loss:.4f}')

In [ ]:
# === Save model ===
torch.save(brains.state_dict(), 'nicto_v5_7brain.pt')
print('Model saved to nicto_v5_7brain.pt')
print(f'Total params: {sum(p.numel() for p in brains.parameters()):,}')